# ATLOP baseline — 약점 정성 테스트

> **최종 업데이트**: 2026-07-12: ATLOP의 알려진 약점 2가지(상호참조 신호 부족, 장거리·복잡 구문 추론 실패)를 임의 문서 각 1건으로 확인하는 probe 셀 추가.

학습이 끝난 ATLOP baseline이 **어떤 유형의 문서에서 실패하는지** 정성적으로 확인한다. 알려진 약점 두 가지:

1. **상호참조(coreference) 신호 부족** — ATLOP은 vertexSet에 명시된 멘션 위치만 logsumexp로 풀링한다. 근거 문장의 주어가 대명사(She/He/It)인데 그 대명사가 멘션으로 등록돼 있지 않으면, 모델이 attention만으로 "She = Marie Curie"를 연결해야 해서 관계를 놓치기 쉽다.
2. **장거리·복잡 구문 추론 실패** — head와 tail이 여러 문장 떨어져 있고 사이에 교란 엔티티가 있거나, 관계 서술이 길게 꼬인 내포절 안에 있으면 Localized Context Pooling이 관련 문맥을 제대로 잡지 못한다.

**사전 조건**: GPU/Colab에서 학습을 마친 체크포인트가 `results/atlop.pt`에 있어야 한다.

```bash
python -m Scripts.atlop.train_re --epochs 30 --distant_limit 20000 --distant_epochs 1 --run_name atlop --save_model
```

체크포인트 없이 실행하면 미학습 가중치라 결과가 무의미하다(경고 출력됨).

In [1]:
# 셋업: 학습된 ATLOP 로드 + probe 헬퍼
# (프로젝트 루트에서 노트북 실행. 학습 시 --emb_size/--block_size를 바꿨다면 아래도 맞출 것)
import sys
from pathlib import Path

import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer

ROOT = Path.cwd()
assert (ROOT / "data").exists(), "프로젝트 루트에서 노트북을 실행하세요"
sys.path.insert(0, str(ROOT))

from data.docred_io import build_rel2id, load_rel_info, NUM_CLASSES
from Scripts.atlop.preprocess import build_features
from Scripts.atlop.re_model import DocREModel
from Scripts.atlop.train_re import make_collate_fn

MODEL_NAME = "bert-base-cased"
CKPT = ROOT / "results" / "atlop.pt"  # train_re.py --save_model 산출물

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)
encoder = AutoModel.from_pretrained(MODEL_NAME, config=config, attn_implementation="eager")
config.cls_token_id = tokenizer.cls_token_id
config.sep_token_id = tokenizer.sep_token_id
model = DocREModel(config, encoder, num_labels=NUM_CLASSES).to(device)

if CKPT.exists():
    model.load_state_dict(torch.load(CKPT, map_location=device))
    print(f"체크포인트 로드 완료: {CKPT}")
else:
    print("[경고] results/atlop.pt 없음 — 미학습 가중치라 결과 무의미. "
          "GPU/Colab에서 --save_model 로 학습 후 체크포인트를 가져올 것.")
model.eval()

rel2id = build_rel2id()
id2rel = {v: k for k, v in rel2id.items()}
rel_names = load_rel_info()  # P-code -> 사람이 읽는 이름
collate = make_collate_fn(tokenizer.pad_token_id)


def probe(doc, pairs_of_interest=None):
    """DocRED 형식 dict 하나를 모델에 통과시켜 예측 triple을 출력.
    pairs_of_interest: 주목할 (h_idx, t_idx) 목록 — 검출 여부를 별도 표시."""
    feats = build_features([doc], tokenizer, rel2id, show_progress=False)
    batch = collate(feats)
    with torch.no_grad():
        preds = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            entity_pos=batch["entity_pos"],
            hts=batch["hts"],
        )[0].cpu()
    names = [v[0]["name"] for v in doc["vertexSet"]]
    found = []
    for (h, t), row in zip(feats[0]["hts"], preds):
        for r in range(1, NUM_CLASSES):
            if row[r] == 1:
                found.append((h, t, id2rel[r]))
    if not found:
        print("예측된 관계 없음 (모두 Na)")
    for h, t, p in found:
        mark = "  <-- 관심 쌍" if pairs_of_interest and (h, t) in pairs_of_interest else ""
        print(f"({names[h]}) --{p}:{rel_names.get(p, '?')}--> ({names[t]}){mark}")
    if pairs_of_interest:
        for h, t in pairs_of_interest:
            if not any(fh == h and ft == t for fh, ft, _ in found):
                print(f"[미검출] ({names[h]}, {names[t]}) 쌍에서 관계를 찾지 못함")
    return found

체크포인트 로드 완료: c:\Users\dohy3\OneDrive\바탕 화면\멋사NLP5기\04_실전프로젝트1\project1\results\atlop.pt


## 테스트 1 — 상호참조 신호 부족

관계의 근거 문장이 head 엔티티를 **대명사로만** 지칭하는 문서. DocRED의 vertexSet은 보통 대명사 멘션까지 포함하지만, 여기서는 일부러 이름 멘션만 등록해서 모델이 순수하게 attention으로 상호참조를 해소해야 하는 상황을 만든다.

- `Marie Curie`의 멘션: 문장 0의 이름뿐 (문장 1의 `She`는 미등록)
- 정답 관계 **P19(place of birth)** 의 근거는 문장 1에만 존재
- 관심 쌍 `(Marie Curie, Warsaw)`가 **미검출**되면 → 약점 1 발현

In [2]:
# 테스트 1: 상호참조(coreference) 신호 부족
# 근거 문장(문장 1)의 주어가 대명사 "She"인데, vertexSet에는 이름 멘션("Marie Curie")만 있음.
# 모델이 attention만으로 "She" = "Marie Curie" 를 연결해야 P19(place of birth)를 찾을 수 있다.
doc_coref = {
    "title": "probe_coref",
    "sents": [
        ["Marie", "Curie", "was", "a", "Polish", "physicist", "."],
        ["She", "was", "born", "in", "Warsaw", "in", "1867", "."],
    ],
    "vertexSet": [
        [{"name": "Marie Curie", "sent_id": 0, "pos": [0, 2], "type": "PER"}],
        [{"name": "Warsaw", "sent_id": 1, "pos": [4, 5], "type": "LOC"}],
    ],
    "labels": [],
}
print("기대: (Marie Curie) --P19:place of birth--> (Warsaw)")
probe(doc_coref, pairs_of_interest=[(0, 1)]);

기대: (Marie Curie) --P19:place of birth--> (Warsaw)
(Marie Curie) --P19:place of birth--> (Warsaw)  <-- 관심 쌍


## 테스트 2 — 장거리·복잡 구문 추론

관계의 근거가 **여러 문장에 걸쳐 있고**, tail 엔티티가 긴 내포절(관계절 + 삽입구) 끝에 있으며, 중간에 같은 타입(PER/MISC)의 교란 엔티티가 배치된 문서. `War and Peace`(문장 0) → `Leo Tolstoy`(문장 3 끝)의 **P50(author)** 을 찾으려면 "the novel published in 1869"가 문장 0의 그 소설임을 장거리로 연결해야 한다.

- 관심 쌍이 **미검출**되거나 교란 쌍(예: Dostoevsky-War and Peace)에 오예측하면 → 약점 2 발현.

In [3]:
# 테스트 2: 장거리 + 꼬인 구문
# head(War and Peace, 문장 0)와 tail(Leo Tolstoy, 문장 3의 긴 내포절 끝)이 멀리 떨어져 있고,
# 사이에 교란 엔티티(다른 작가/작품)가 끼어 있으며, 연결 고리("the novel published in 1869")가
# 관계절 안에 숨어 있다.
doc_long = {
    "title": "probe_long_distance",
    "sents": [
        ["War", "and", "Peace", "was", "published", "in", "1869", "."],
        ["Fyodor", "Dostoevsky", "wrote", "Crime", "and", "Punishment", "."],
        ["Anton", "Chekhov", "is", "known", "for", "his", "short", "stories", "."],
        ["Many", "critics", ",", "who", "had", "for", "decades", "questioned",
         "whether", "any", "single", "novel", "could", "define", "Russian",
         "literature", ",", "eventually", "agreed", "that", "the", "novel",
         "published", "in", "1869", "was", "the", "masterpiece", "of",
         "Leo", "Tolstoy", "."],
    ],
    "vertexSet": [
        [{"name": "War and Peace", "sent_id": 0, "pos": [0, 3], "type": "MISC"}],
        [{"name": "Leo Tolstoy", "sent_id": 3, "pos": [29, 31], "type": "PER"}],
        [{"name": "Fyodor Dostoevsky", "sent_id": 1, "pos": [0, 2], "type": "PER"}],
        [{"name": "Crime and Punishment", "sent_id": 1, "pos": [3, 6], "type": "MISC"}],
        [{"name": "Anton Chekhov", "sent_id": 2, "pos": [0, 2], "type": "PER"}],
    ],
    "labels": [],
}
print("기대: (War and Peace) --P50:author--> (Leo Tolstoy)")
probe(doc_long, pairs_of_interest=[(0, 1)]);

기대: (War and Peace) --P50:author--> (Leo Tolstoy)
(War and Peace) --P50:author--> (Fyodor Dostoevsky)
(Crime and Punishment) --P50:author--> (Fyodor Dostoevsky)
(Crime and Punishment) --P50:author--> (Anton Chekhov)
[미검출] (War and Peace, Leo Tolstoy) 쌍에서 관계를 찾지 못함
